In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import median_abs_deviation
import gseapy

names = gseapy.get_library_name()
print(names)

### Load data

In [ ]:
adata = sc.read_h5ad('../yourpath.h5ad')
adata

### SERPINE1

In [ ]:
adata_sub = adata[adata.obs['anno_sub'].isin(['Cancer cells (classical)','Cancer cells (basal-like)','Cancer cells (exocrine-like)'])].copy()
adata_sub = adata_sub[adata_sub.obs['tissue'].isin(['tumor'])].copy()
adata_sub.obs['anno_sub'] = adata_sub.obs['anno_sub'].replace({
    'Cancer cells (basal-like)': 'Cancer cells',
    'Cancer cells (classical)' : 'Cancer cells',
    'Cancer cells (exocrine-like)': 'Cancer cells'})
adata_sub.obs

v = adata_sub.obs_vector('SERPINE1')
adata_sub.obs['SERPINE1_pos'] = pd.Categorical(
    np.where(v > 0, 'pos', 'neg'),
    categories=['neg', 'pos']
)

#Filter genes
cluster_tissue_matrix = pd.crosstab(adata_sub.obs['disease'], adata_sub.obs['SERPINE1_pos'])
print(cluster_tissue_matrix)
mt_genes = adata_sub.var_names.str.startswith('MT-')
ribo_genes = adata_sub.var_names.str.startswith(('RPS', 'RPL'))
adata_sub = adata_sub[:, ~(mt_genes | ribo_genes)].copy()
smallest_population = cluster_tissue_matrix.min().min()
min_cells = int(0.05 * smallest_population)
sc.pp.filter_genes(adata_sub, min_cells=min_cells)
print(f"Data cleaned. Remaining genes: {adata_sub.n_vars}, Remaining cells: {adata_sub.n_obs}")

In [ ]:
adata_sub.raw = None
sc.tl.rank_genes_groups(adata_sub, 'SERPINE1_pos', method='wilcoxon', use_raw=False)
sc.pl.rank_genes_groups(adata_sub, n_genes=25, sharey=False)

In [ ]:
results = adata_sub.uns['rank_genes_groups']
out = np.array([[0,0,0,0,0]])
for group in results['names'].dtype.names:
    out = np.vstack((out, np.vstack((results['names'][group],
                                     results['scores'][group],
                                     results['pvals_adj'][group],
                                     results['logfoldchanges'][group],
                                     np.array([group]*len(results['names'][group])).astype('object'))).T))
    
markers = pd.DataFrame(out[1:], columns = ['Gene', 'scores', 'pval_adj', 'lfc', 'cluster'])
markers = markers[(markers.pval_adj < 0.05) & (markers.lfc > 1) & (markers.cluster == "pos")]
markers.to_excel('../yourpath/SERPINE1_cleaned_enrichR_human_up_padj0.05_lfc1/wilcoxon.deg.xlsx')
markers = list(markers['Gene'])

In [ ]:
enr_GOBP_up = gseapy.enrichr(gene_list=markers ,
 gene_sets=['Reactome_Pathways_2024'],
 organism='Human', 
 outdir='../yourpath/SERPINE1_cleaned_enrichR_human_up_padj0.05_lfc1/',
 cutoff=0.5 
 )

### SERPINB2

In [ ]:
adata_sub = adata[adata.obs['anno_sub'].isin(['Cancer cells (classical)','Cancer cells (basal-like)','Cancer cells (exocrine-like)'])].copy()
adata_sub = adata_sub[adata_sub.obs['tissue'].isin(['tumor'])].copy()
adata_sub.obs['anno_sub'] = adata_sub.obs['anno_sub'].replace({
    'Cancer cells (basal-like)': 'Cancer cells',
    'Cancer cells (classical)' : 'Cancer cells',
    'Cancer cells (exocrine-like)': 'Cancer cells'})
adata_sub.obs

v = adata_sub.obs_vector('SERPINB2')
adata_sub.obs['SERPINB2_pos'] = pd.Categorical(
    np.where(v > 0, 'pos', 'neg'),
    categories=['neg', 'pos']
)

#Filter genes
cluster_tissue_matrix = pd.crosstab(adata_sub.obs['disease'], adata_sub.obs['SERPINB2_pos'])
print(cluster_tissue_matrix)
mt_genes = adata_sub.var_names.str.startswith('MT-')
ribo_genes = adata_sub.var_names.str.startswith(('RPS', 'RPL'))
adata_sub = adata_sub[:, ~(mt_genes | ribo_genes)].copy()
smallest_population = cluster_tissue_matrix.min().min()
min_cells = int(0.05 * smallest_population)
sc.pp.filter_genes(adata_sub, min_cells=min_cells)
print(f"Data cleaned. Remaining genes: {adata_sub.n_vars}, Remaining cells: {adata_sub.n_obs}")

In [ ]:
adata_sub.raw = None
sc.tl.rank_genes_groups(adata_sub, 'SERPINB2_pos', method='wilcoxon', use_raw=False)
sc.pl.rank_genes_groups(adata_sub, n_genes=25, sharey=False)

In [ ]:
results = adata_sub.uns['rank_genes_groups']
out = np.array([[0,0,0,0,0]])
for group in results['names'].dtype.names:
    out = np.vstack((out, np.vstack((results['names'][group],
                                     results['scores'][group],
                                     results['pvals_adj'][group],
                                     results['logfoldchanges'][group],
                                     np.array([group]*len(results['names'][group])).astype('object'))).T))
    
markers = pd.DataFrame(out[1:], columns = ['Gene', 'scores', 'pval_adj', 'lfc', 'cluster'])
markers = markers[(markers.pval_adj < 0.05) & (markers.lfc > 1.5) & (markers.cluster == "pos")]
markers.to_excel('../yourpath/SERPINB2_cleaned_enrichR_human_up_padj0.05_lfc1/wilcoxon.deg.xlsx')
markers = list(markers['Gene'])

In [ ]:
enr_GOBP_up = gseapy.enrichr(gene_list=markers ,
 gene_sets=['Reactome_Pathways_2024'],
 organism='Human', 
 outdir='../yourpath/SERPINB2_cleaned_enrichR_human_up_padj0.05_lfc1/',
 cutoff=0.5 
 )